In [1]:
# 1. 라이브러리 및 경로 설정
from ultralytics import YOLO
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "dataset.yaml").exists():
    ROOT = ROOT.parent
if not (ROOT / "dataset.yaml").exists():
    raise RuntimeError("Run from the repository root or notebooks directory.")
DATA_YAML = ROOT / "dataset.yaml"
RESULT_ROOT = ROOT / "results"

In [2]:
# 2. 실험 함수 정의

def run_experiments(model_name, epochs_list):
    results_summary = []

    for epochs in epochs_list:
        exp_name = f"{model_name.replace('.pt', '')}_ep{epochs}"

        print(f"\n{'='*40}")
        print(f"실험 시작: {exp_name}")
        print(f"{'='*40}")

        # 학습
        model = YOLO(str(ROOT / "weights" / model_name))
        model.train(
            data=str(DATA_YAML),
            epochs=epochs,
            imgsz=640,
            batch=16,
            device=0,
            project=str(RESULT_ROOT),
            name=f"{exp_name}_train",
            exist_ok=True
        )

        # best.pt 경로
        best_pt = RESULT_ROOT / f"{exp_name}_train" / "weights" / "best.pt"

        # test 평가
        model = YOLO(str(best_pt))
        test_result = model.val(
            data=str(DATA_YAML),
            split="test",
            device=0,
            project=str(RESULT_ROOT),
            name=f"{exp_name}_test",
            exist_ok=True
        )

        results_summary.append({
            "name": exp_name,
            "mAP50": round(test_result.box.map50, 3),
            "mAP50-95": round(test_result.box.map, 3),
            "precision": round(test_result.box.mp, 3),
            "recall": round(test_result.box.mr, 3),
            "normal_mAP50": round(test_result.box.ap50[0], 3),
            "abnormal_mAP50": round(test_result.box.ap50[1], 3),
            "normal_mAP50-95": round(test_result.box.ap[0], 3),
            "abnormal_mAP50-95": round(test_result.box.ap[1], 3),
        })

    # 결과 비교 출력
    print(f"\n\n===== {model_name} 실험 결과 =====")
    print(f"{'모델':<20} {'mAP50':>8} {'mAP50-95':>10} {'P':>8} {'R':>8} {'normal_50':>10} {'abnormal_50':>12} {'normal_95':>10} {'abnormal_95':>12}")
    print("-" * 100)
    for r in results_summary:
        print(f"{r['name']:<20} {r['mAP50']:>8} {r['mAP50-95']:>10} {r['precision']:>8} {r['recall']:>8} {r['normal_mAP50']:>10} {r['abnormal_mAP50']:>12} {r['normal_mAP50-95']:>10} {r['abnormal_mAP50-95']:>12}")

    return results_summary

In [ ]:
# 3. RT-DETR 실험 
import torch
torch.cuda.empty_cache()

def run_rtdetr_experiments(model_name, epochs_list):
    results_summary = []

    for epochs in epochs_list:
        exp_name = f"{model_name.replace('.pt', '')}_ep{epochs}"

        print(f"\n{'='*40}")
        print(f"실험 시작: {exp_name}")
        print(f"{'='*40}")

        model = YOLO(str(ROOT / "weights" / model_name))
        model.train(
            data=str(DATA_YAML),
            epochs=epochs,
            imgsz=640,
            batch=8,          # batch 줄임
            device=0,
            project=str(RESULT_ROOT),
            name=f"{exp_name}_train",
            exist_ok=True
        )

        best_pt = RESULT_ROOT / f"{exp_name}_train" / "weights" / "best.pt"

        model = YOLO(str(best_pt))
        test_result = model.val(
            data=str(DATA_YAML),
            split="test",
            device=0,
            project=str(RESULT_ROOT),
            name=f"{exp_name}_test",
            exist_ok=True
        )

        f1 = round(2 * test_result.box.mp * test_result.box.mr /
                   (test_result.box.mp + test_result.box.mr + 1e-8), 3)

        results_summary.append({
            "name": exp_name,
            "mAP50": round(test_result.box.map50, 3),
            "mAP50-95": round(test_result.box.map, 3),
            "precision": round(test_result.box.mp, 3),
            "recall": round(test_result.box.mr, 3),
            "normal_mAP50": round(test_result.box.ap50[0], 3),
            "abnormal_mAP50": round(test_result.box.ap50[1], 3),
            "normal_mAP50-95": round(test_result.box.ap[0], 3),
            "abnormal_mAP50-95": round(test_result.box.ap[1], 3),
        })

    print(f"\n\n===== {model_name} 실험 결과 =====")
    print(f"{'모델':<20} {'mAP50':>8} {'mAP50-95':>10} {'P':>8} {'R':>8} {'normal_50':>10} {'abnormal_50':>12} {'normal_95':>10} {'abnormal_95':>12}")
    print("-" * 100)
    for r in results_summary:
        print(f"{r['name']:<20} {r['mAP50']:>8} {r['mAP50-95']:>10} {r['precision']:>8} {r['recall']:>8} {r['normal_mAP50']:>10} {r['abnormal_mAP50']:>12} {r['normal_mAP50-95']:>10} {r['abnormal_mAP50-95']:>12}")

    return results_summary

run_rtdetr_experiments("rtdetr-l.pt", [20, 25, 30])


실험 시작: rtdetr-l_ep20
New https://pypi.org/project/ultralytics/8.4.56 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.48 🚀 Python-3.13.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080, 9875MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/data1/workspaces/tp2026/scooter-anomaly-detection/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mode

/home/tp2026/miniconda3/lib/python3.13/site-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/20      6.61G     0.8712       1.37      1.332         12        640: 100% ━━━━━━━━━━━━ 177/177 1.7it/s 1:460.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 3.0it/s 8.6s0.2s
                   all        403        447      0.221      0.258      0.155     0.0659

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/20      6.78G     0.4915      1.062     0.6518         19        640: 0% ──────────── 0/177  0.5s

/home/tp2026/miniconda3/lib/python3.13/site-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/20      6.79G     0.5079       1.03     0.5836          6        640: 100% ━━━━━━━━━━━━ 177/177 2.0it/s 1:270.5sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 5.2it/s 5.0s0.2s
                   all        403        447      0.433      0.336      0.276      0.131

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/20      6.91G     0.4504      0.866     0.5503         19        640: 0% ──────────── 0/177  0.5s

/home/tp2026/miniconda3/lib/python3.13/site-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (142 outputs left out)

      29/30       6.9G     0.2657     0.3674     0.3955          5        640: 100% ━━━━━━━━━━━━ 177/177 2.0it/s 1:280.5sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 5.1it/s 5.1s0.2s
                   all        403        447       0.94      0.933      0.956      0.682

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/30       6.8G     0.1685     0.3383     0.1789         11        640: 0% ──────────── 0/177  0.5s

/home/tp2026/miniconda3/lib/python3.13/site-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30       6.8G      0.255     0.3508     0.3751          6        640: 100% ━━━━━━━━━━━━ 177/177 2.0it/s 1:280.5sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 5.1it/s 5.1s0.2s
                   all        403        447      0.952      0.918      0.958      0.684

30 epochs completed in 0.826 hours.
Optimizer stripped from /data1/workspaces/tp2026/scooter-anomaly-detection/results/rtdetr-l_ep30_train/weights/last.pt, 66.2MB
Optimizer stripped from /data1/workspaces/tp2026/scooter-anomaly-detection/results/rtdetr-l_ep30_train/weights/best.pt, 66.2MB

Validating /data1/workspaces/tp2026/scooter-anomaly-detection/results/rtdetr-l_ep30_train/weights/best.pt...
Ultralytics 8.4.48 🚀 Python-3.13.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080, 9875MiB)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50 

[{'name': 'rtdetr-l_ep20',
  'mAP50': np.float64(0.91),
  'mAP50-95': np.float64(0.646),
  'precision': np.float64(0.884),
  'recall': np.float64(0.832),
  'normal_mAP50': np.float64(0.904),
  'abnormal_mAP50': np.float64(0.917),
  'normal_mAP50-95': np.float64(0.632),
  'abnormal_mAP50-95': np.float64(0.661)},
 {'name': 'rtdetr-l_ep25',
  'mAP50': np.float64(0.918),
  'mAP50-95': np.float64(0.677),
  'precision': np.float64(0.903),
  'recall': np.float64(0.895),
  'normal_mAP50': np.float64(0.934),
  'abnormal_mAP50': np.float64(0.902),
  'normal_mAP50-95': np.float64(0.668),
  'abnormal_mAP50-95': np.float64(0.685)},
 {'name': 'rtdetr-l_ep30',
  'mAP50': np.float64(0.921),
  'mAP50-95': np.float64(0.675),
  'precision': np.float64(0.935),
  'recall': np.float64(0.909),
  'normal_mAP50': np.float64(0.927),
  'abnormal_mAP50': np.float64(0.914),
  'normal_mAP50-95': np.float64(0.665),
  'abnormal_mAP50-95': np.float64(0.686)}]